# Task 14: Custom CUDA Kernel Integration for Accelerated Activation Functions

## Objective

To implement a custom CUDA kernel for the SwiGLU activation function and compare its execution with a standard PyTorch implementation.

## Technologies / Tools Used

- Python 3.10+
- C++
- CUDA Toolkit
- PyTorch
- PyTorch C++ Extensions
- Google Colab

## Formula

### SiLU Activation

SiLU(x) = x × sigmoid(x)

### SwiGLU

SwiGLU(x, gate) = SiLU(gate) × x

The CUDA kernel performs these operations directly on GPU threads.

## Step 1: Check CUDA Availability

The custom CUDA kernel requires a CUDA-enabled GPU.

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Please enable GPU in Colab: Runtime → Change runtime type → GPU")

PyTorch version: 2.9.0+cpu
CUDA available: False
Please enable GPU in Colab: Runtime → Change runtime type → GPU


## Step 2: Create the CUDA Kernel

Create a CUDA kernel where each GPU thread processes one element of the input.

In [2]:
# Create CUDA source code

cuda_source = r'''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

__global__ void swiglu_kernel(
    const float* x,
    const float* gate,
    float* output,
    int n
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n) {

        float g = gate[i];

        float sigmoid = 1.0f /
            (1.0f + expf(-g));

        output[i] = x[i] * g * sigmoid;
    }
}
'''

print("CUDA kernel created.")

CUDA kernel created.


## Step 3: Write and Compile the CUDA Extension

Compile the custom CUDA implementation using PyTorch's extension mechanism.

In [3]:
from torch.utils.cpp_extension import load_inline

if torch.cuda.is_available():

    cpp_source = r'''
    #include <torch/extension.h>

    void swiglu_cuda(
        torch::Tensor x,
        torch::Tensor gate,
        torch::Tensor output
    );
    '''

    cuda_source_full = cuda_source + r'''
    void swiglu_cuda(
        torch::Tensor x,
        torch::Tensor gate,
        torch::Tensor output
    ) {

        int n = x.numel();

        int threads = 256;

        int blocks =
            (n + threads - 1) / threads;

        swiglu_kernel<<<blocks, threads>>>(
            x.data_ptr<float>(),
            gate.data_ptr<float>(),
            output.data_ptr<float>(),
            n
        );
    }

    PYBIND11_MODULE(
        TORCH_EXTENSION_NAME,
        m
    ) {
        m.def(
            "swiglu_cuda",
            &swiglu_cuda,
            "SwiGLU CUDA"
        );
    }
    '''

    swiglu_cuda = load_inline(
        name="swiglu_cuda",
        cpp_sources=cpp_source,
        cuda_sources=cuda_source_full,
        functions=None,
        verbose=False
    )

    print("CUDA extension compiled successfully.")

else:

    print("CUDA GPU not available.")

CUDA GPU not available.


## Step 4: Benchmark the CUDA Kernel

Compare the custom CUDA implementation with the standard PyTorch implementation.

In [4]:
if torch.cuda.is_available():

    # Create test tensors

    x = torch.randn(
        1000000,
        device="cuda"
    )

    gate = torch.randn(
        1000000,
        device="cuda"
    )

    output = torch.empty_like(x)

    # Standard PyTorch

    torch.cuda.synchronize()

    start = torch.cuda.Event(
        enable_timing=True
    )

    end = torch.cuda.Event(
        enable_timing=True
    )

    start.record()

    for _ in range(100):
        standard = x * torch.nn.functional.silu(gate)

    end.record()

    torch.cuda.synchronize()

    pytorch_time = start.elapsed_time(end)

    # Custom CUDA

    start.record()

    for _ in range(100):
        swiglu_cuda.swiglu_cuda(
            x,
            gate,
            output
        )

    end.record()

    torch.cuda.synchronize()

    cuda_time = start.elapsed_time(end)

    print("PyTorch time:", pytorch_time, "ms")
    print("Custom CUDA time:", cuda_time, "ms")

    print(
        "Speedup:",
        pytorch_time / cuda_time
    )

else:

    print("CUDA benchmark skipped because GPU is unavailable.")

CUDA benchmark skipped because GPU is unavailable.


## Conclusion

A custom CUDA kernel for the SwiGLU activation function was created and compiled using PyTorch C++ extensions. The implementation uses GPU threads to perform the activation and provides a basis for comparing custom GPU kernels with standard PyTorch operations.